In [2]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns


## Week 1

In [3]:
# Load the dataset
df_nr = pd.read_csv('/Users/gaurav/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_1.2.csv')
df_nr.head()

,X,Y,Z
0,0,0.548814,0.548814
1,1,1.215189,0.715189
2,0,0.602763,0.602763
3,0,0.544883,0.544883
4,0,0.423655,0.423655


In [5]:
# 2. Separate into Treatment (X = 1) and Control (X = 0) groups
treatment_group = df_nr[df_nr['X'] == 1].copy()
control_group = df_nr[df_nr['X'] == 0].copy()

In [6]:
# 3. Fit a nearest neighbor model on the control group
nn_model = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn_model.fit(control_group[['Z']])


,n_neighbors,1
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


In [7]:
# 4. Find the closest X = 0 match for each sample in X = 1
distances, indices = nn_model.kneighbors(treatment_group[['Z']])

In [8]:
# 5. Map match results back to the treated group dataframe
treatment_group['matched_control_index'] = control_group.index[indices.flatten()]
treatment_group['matched_control_Z'] = control_group['Z'].iloc[indices.flatten()].values
treatment_group['match_distance'] = distances.flatten()

# 6. Display the treatment group with matched control information
print(treatment_group[['X', 'Z', 'matched_control_index', 'matched_control_Z', 'match_distance']])    

    X         Z  matched_control_index  matched_control_Z  match_distance
1   1  0.715189                     93           0.716327        0.001138
5   1  0.645894                     56           0.653108        0.007214
6   1  0.437587                     41           0.437032        0.000555
7   1  0.891773                     18           0.778157        0.113616
8   1  0.963663                     18           0.778157        0.185506
9   1  0.383442                     29           0.414662        0.031220
10  1  0.791725                     18           0.778157        0.013568
13  1  0.925597                     18           0.778157        0.147440
17  1  0.832620                     18           0.778157        0.054463
19  1  0.870012                     18           0.778157        0.091855
20  1  0.978618                     18           0.778157        0.200462
21  1  0.799159                     18           0.778157        0.021002
22  1  0.461479                     58

In [9]:
treatment_group['match_distance'].describe()

count    48.000000
mean      0.049788
std       0.068388
min       0.000389
25%       0.002509
50%       0.013260
75%       0.055794
max       0.210217
Name: match_distance, dtype: float64

In [10]:
# The maximum distance among all matched pairs
farthest_distance = distances.max()
# 5. Extract the matched X = 0 records
matched_control_indices = control_group.index[indices.flatten()]
matched_control = control_group.loc[matched_control_indices]

# Calculate avarage Y values
mean_Y_treated = treatment_group['Y'].mean()
mean_Y_matched_control = matched_control['Y'].mean()

# Effect calculations
effect_control_minus_treated = mean_Y_matched_control - mean_Y_treated
effect_treated_minus_control = mean_Y_treated - mean_Y_matched_control
farthest_distance = distances.max()

print(f"Farthest Match Distance: {farthest_distance:.4f}")
print(f"Mean Y (X = 1): {mean_Y_treated:.4f}")
print(f"Mean Y (Matched X = 0): {mean_Y_matched_control:.4f}")
print(f"Effect (Mean Y_X=0 - Mean Y_X=1): {effect_control_minus_treated:.4f}")
print(f"Effect (Mean Y_X=1 - Mean Y_X=0): {effect_treated_minus_control:.4f}")

Farthest Match Distance: 0.2102
Mean Y (X = 1): 1.1256
Mean Y (Matched X = 0): 0.5822
Effect (Mean Y_X=0 - Mean Y_X=1): -0.5434
Effect (Mean Y_X=1 - Mean Y_X=0): 0.5434


----

1. In Coding Quiz 1, you are asked to find the distance of the farthest match in a set.  Is this farthest match distance too far to be a meaningful match?  How can you decide this?

## Answer to Question 1 — Is the farthest match (0.2102) too far?

**Short answer:** the number 0.2102 alone does not prove the match is meaningless, but in this notebook it flags genuinely weak matches. The right response is to judge it against the distance distribution, the Z scale, the overlap of the two groups, and the size of the effect — not by gut feeling.

**Evidence from the notebook:**

- The 48 match distances: median 0.0133, 75th percentile 0.0558, mean 0.0498, max 0.2102. The worst match is ~16x the median and ~4x the mean — an outlier, not a typical match.
- 18 of the 48 treated units have Z above the largest control Z (0.7782, control row 18), so they all pile onto that one control. That is a lack of common support at the top end, not just one unlucky draw.
- Z spans roughly 0–1, so 0.21 is ~21% of the entire range.
- Y moves about 1-for-1 with Z here (corr(Z, Y) ≈ 1.0 in both groups), so a 0.21 Z-gap injects ≈ 0.21 of error into that pair's Y comparison — against an estimated effect of ≈ 0.54–0.58. For the worst pairs the mismatch is on the order of 40% of the effect being estimated. That is large.

**How to decide in general:**

1. **Caliper:** pre-specify a maximum acceptable distance (e.g. 0.02, 0.05, 0.10) grounded in how much Z matters for Y; drop or flag matches beyond it. (Approach B's 0.2 radius is exactly this idea.)
2. **Compare the max to the distribution** (median, quartiles): an extreme max with a tiny median means a few bad matches, not global failure.
3. **Check overlap / common support:** treated units outside the control Z range cannot be well matched by construction.
4. **Sensitivity check:** re-estimate the effect dropping the worst matches; if it barely moves, the far matches were not driving the result.
5. **Balance checks:** standardized mean differences in Z before vs. after matching.

**Verdict for this quiz:** the farthest match (0.2102: treated Z ≈ 0.988 vs. control Z ≈ 0.778) is a poor match that would fail a strict caliper, along with a handful of other high-Z pairings (≈ 0.15–0.20). But overall matching quality is good (median 0.013), so the conclusion is to flag/drop the few unsupported high-Z pairs and run a sensitivity check — not to discard matching entirely.
